<a href="https://colab.research.google.com/github/ncastro24/simplify-med-docs/blob/main/notebooks/qlora_phi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes evaluate

In [ ]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

In [ ]:
from google.colab import files
files.upload()

In [ ]:
dataset = load_dataset("json", data_files="biomed_dataset.json")["train"]

dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(len(train_dataset), len(eval_dataset))

In [ ]:
model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def safe_join(x):
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        return "\n".join([str(i) for i in x])
    return str(x)


def format_example(example):
    return (
        f"{example['instruction']}\n"
        f"Input: {safe_join(example['input'])}\n"
        f"Output: {safe_join(example['output'])}"
    )

In [ ]:
MAX_LEN = 128

def tokenize(example):
    text = format_example(example)

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_dataset = train_dataset.map(tokenize)
eval_dataset = eval_dataset.map(tokenize)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu_score = bleu.compute(
        predictions=[p.split() for p in decoded_preds],
        references=[[l.split()] for l in decoded_labels]
    )

    rouge_score = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "bleu": bleu_score["bleu"],
        "rouge1": rouge_score["rouge1"],
        "rouge2": rouge_score["rouge2"],
        "rougeL": rouge_score["rougeL"]
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=3,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    logging_steps=5,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("phi2-biomed-qlora")
tokenizer.save_pretrained("phi2-biomed-qlora")

In [ ]:
results = trainer.evaluate()

df_results = pd.DataFrame([results])
df_results

In [ ]:
df_results.to_csv("evaluation_results.csv", index=False)

In [ ]:
log_history = trainer.state.log_history

train_loss = []
eval_loss = []
steps = []

for entry in log_history:
    if "loss" in entry:
        train_loss.append(entry["loss"])
        steps.append(entry["step"])

    if "eval_loss" in entry:
        eval_loss.append(entry["eval_loss"])


plt.plot(steps[:len(train_loss)], train_loss, label="train loss")
plt.title("Training Loss")
plt.legend()
plt.show()

In [ ]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    device_map="auto"
)

fine_tuned_model = model  # your trained QLoRA model

In [ ]:
test_set = eval_dataset  # or a held-out test split

In [ ]:
def generate_output(model, example):

    text = format_example(example)

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
base_preds = []
ft_preds = []
refs = []

for ex in test_set:

    base_out = generate_output(base_model, ex)
    ft_out = generate_output(fine_tuned_model, ex)

    base_preds.append(base_out)
    ft_preds.append(ft_out)

    refs.append(safe_join(ex["output"]))

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")


def compute(preds):
    return {
        "bleu": bleu.compute(
            predictions=[p.split() for p in preds],
            references=[[r.split()] for r in refs]
        )["bleu"],

        "rouge": rouge.compute(
            predictions=preds,
            references=refs
        )["rougeL"]
    }


base_scores = compute(base_preds)
ft_scores = compute(ft_preds)

In [ ]:
import pandas as pd

results_table = pd.DataFrame([
    {
        "Model": "Base (Phi-2)",
        "BLEU": base_scores["bleu"],
        "ROUGE-L": base_scores["rouge"]
    },
    {
        "Model": "Fine-tuned (QLoRA)",
        "BLEU": ft_scores["bleu"],
        "ROUGE-L": ft_scores["rouge"]
    }
])

results_table

In [ ]:
results_table.to_csv("model_comparison_results.csv", index=False)

In [ ]:
rows = []

for ex in test_set:

    input_text = format_example(ex)

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    # -------------------------
    # Base model output
    # -------------------------
    base_outputs = base_model.generate(**inputs, max_new_tokens=100)
    base_text = tokenizer.decode(base_outputs[0], skip_special_tokens=True)

    # -------------------------
    # Fine-tuned model output
    # -------------------------
    ft_outputs = fine_tuned_model.generate(**inputs, max_new_tokens=100)
    ft_text = tokenizer.decode(ft_outputs[0], skip_special_tokens=True)

    # -------------------------
    # Reference (ground truth)
    # -------------------------
    reference = safe_join(ex["output"])

    # -------------------------
    # Store row
    # -------------------------
    rows.append({
        "Input": safe_join(ex["input"]),
        "Base Output": base_text,
        "Fine-tuned Output": ft_text,
        "Reference": reference
    })

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(rows)

comparison_df

In [ ]:
comparison_df.to_csv("per_example_comparison.csv", index=False)

In [ ]:
def test(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(test("Simplify: The patient presents with dyspnea and tachycardia."))

Final saving of outputs

In [ ]:
model.save_pretrained("phi2-biomed-qlora")
tokenizer.save_pretrained("phi2-biomed-qlora")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/phi2-biomed-qlora")
tokenizer.save_pretrained("/content/drive/MyDrive/phi2-biomed-qlora")

In [ ]:
comparison_df.to_csv("per_example_comparison.csv", index=False)
results_table.to_csv("model_comparison_results.csv", index=False)

In [ ]:
from google.colab import files
files.download("per_example_comparison.csv")
files.download("model_comparison_results.csv")

In [ ]:
plt.savefig("loss_curve.png")

In [ ]:
files.download("loss_curve.png")